# Crecimiento Mensual
**Proyecto:** Reserva Inteligente de Restaurantes — Etapa 3  
**Análisis:** MoM, YTD, tasa de cancelación y crecimiento por tipo de pedido  
**Fuente:** Data Warehouse Hive (`restaurant_dw`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("crecimiento_mensual_notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE restaurant_dw")
print("Spark version:", spark.version)

In [ ]:
# fact_pedido: select explícito para evitar colisión con columnas de partición anio/mes
fact_pedido = spark.table("fact_pedido").select(
    "id_tiempo", "id_restaurante", "id_usuario",
    "id_tipo_pedido", "id_estado_pedido",
    "id_pedido_origen", "subtotal", "precio_total_pedido",
)

# Renombrar id y nombre en dims para evitar ambigüedad
dim_tiempo      = spark.table("dim_tiempo").withColumnRenamed("id", "id_tiempo")
dim_restaurante = spark.table("dim_restaurante").withColumnRenamed("id", "id_restaurante").withColumnRenamed("nombre", "nombre_restaurante")
dim_estado      = spark.table("dim_estado_pedido").withColumnRenamed("id", "id_estado_pedido").withColumnRenamed("nombre", "estado_nombre")
dim_tipo        = spark.table("dim_tipo_pedido").withColumnRenamed("id", "id_tipo_pedido").withColumnRenamed("nombre", "tipo_nombre")

print(f"fact_pedido: {fact_pedido.count()} filas")

## 1. Crecimiento MoM — pedidos, clientes e ingresos

In [ ]:
df_base = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .join(dim_estado,      "id_estado_pedido")
    .filter(F.col("estado_nombre") == "completado")
    .groupBy(
        F.col("anio"), F.col("mes"), F.col("nombre_mes"),
        F.col("nombre_restaurante").alias("restaurante"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.countDistinct("id_usuario").alias("clientes_unicos"),
        F.round(F.sum("precio_total_pedido"), 2).alias("ingresos_totales"),
        F.round(F.avg("precio_total_pedido"), 2).alias("ticket_promedio"),
    )
)

w = Window.partitionBy("restaurante").orderBy("anio", "mes")

df_mom = (
    df_base
    .withColumn("pedidos_anterior",  F.lag("total_pedidos",    1).over(w))
    .withColumn("clientes_anterior", F.lag("clientes_unicos",  1).over(w))
    .withColumn("ingresos_anterior", F.lag("ingresos_totales", 1).over(w))
    .withColumn("crec_pedidos_pct",
        F.when(F.col("pedidos_anterior") > 0,
            F.round((F.col("total_pedidos") - F.col("pedidos_anterior"))
                    * 100.0 / F.col("pedidos_anterior"), 2)
        ).otherwise(F.lit(None)))
    .withColumn("crec_clientes_pct",
        F.when(F.col("clientes_anterior") > 0,
            F.round((F.col("clientes_unicos") - F.col("clientes_anterior"))
                    * 100.0 / F.col("clientes_anterior"), 2)
        ).otherwise(F.lit(None)))
    .withColumn("crec_ingresos_pct",
        F.when(F.col("ingresos_anterior") > 0,
            F.round((F.col("ingresos_totales") - F.col("ingresos_anterior"))
                    * 100.0 / F.col("ingresos_anterior"), 2)
        ).otherwise(F.lit(None)))
    .drop("pedidos_anterior", "clientes_anterior", "ingresos_anterior")
    .orderBy("restaurante", "anio", "mes")
)

df_mom.show(20, truncate=False)

## 2. Crecimiento acumulado YTD

In [ ]:
w_ytd  = Window.partitionBy("restaurante", "anio").orderBy("mes") \
               .rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_anio = Window.partitionBy("restaurante", "mes").orderBy("anio")

df_ytd = (
    df_base
    .withColumn("pedidos_ytd",  F.sum("total_pedidos").over(w_ytd))
    .withColumn("ingresos_ytd", F.round(F.sum("ingresos_totales").over(w_ytd), 2))
    .withColumn("pedidos_ytd_ant",  F.lag("pedidos_ytd",  1).over(w_anio))
    .withColumn("ingresos_ytd_ant", F.lag("ingresos_ytd", 1).over(w_anio))
    .withColumn("crec_pedidos_ytd_pct",
        F.when(F.col("pedidos_ytd_ant") > 0,
            F.round((F.col("pedidos_ytd") - F.col("pedidos_ytd_ant"))
                    * 100.0 / F.col("pedidos_ytd_ant"), 2)
        ).otherwise(F.lit(None)))
    .withColumn("crec_ingresos_ytd_pct",
        F.when(F.col("ingresos_ytd_ant") > 0,
            F.round((F.col("ingresos_ytd") - F.col("ingresos_ytd_ant"))
                    * 100.0 / F.col("ingresos_ytd_ant"), 2)
        ).otherwise(F.lit(None)))
    .drop("pedidos_ytd_ant", "ingresos_ytd_ant")
    .orderBy("restaurante", "anio", "mes")
)

df_ytd.select(
    "restaurante", "anio", "mes", "nombre_mes",
    "pedidos_ytd", "ingresos_ytd",
    "crec_pedidos_ytd_pct", "crec_ingresos_ytd_pct"
).show(20, truncate=False)

## 3. Tasa de cancelación mensual

In [ ]:
df_estados = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .join(dim_estado,      "id_estado_pedido")
    .groupBy(
        F.col("anio"), F.col("mes"), F.col("nombre_mes"),
        F.col("nombre_restaurante").alias("restaurante"),
        F.col("estado_nombre").alias("estado"),
    )
    .agg(F.countDistinct("id_pedido_origen").alias("total"))
)

df_cancel = (
    df_estados
    .groupBy("anio", "mes", "nombre_mes", "restaurante")
    .pivot("estado", ["completado", "cancelado"])
    .sum("total")
    .withColumnRenamed("completado", "completados")
    .withColumnRenamed("cancelado",  "cancelados")
    .fillna(0)
    .withColumn("total", F.col("completados") + F.col("cancelados"))
    .withColumn("tasa_cancelacion_pct",
        F.when(F.col("total") > 0,
            F.round(F.col("cancelados") * 100.0 / F.col("total"), 2)
        ).otherwise(F.lit(0.0)))
    .orderBy("restaurante", "anio", "mes")
)

df_cancel.show(20, truncate=False)

## 4. Crecimiento por tipo de pedido

In [ ]:
df_tipo = (
    fact_pedido
    .join(dim_tiempo,      "id_tiempo")
    .join(dim_restaurante, "id_restaurante")
    .join(dim_tipo,        "id_tipo_pedido")
    .join(dim_estado,      "id_estado_pedido")
    .filter(F.col("estado_nombre") == "completado")
    .groupBy(
        F.col("anio"), F.col("mes"), F.col("nombre_mes"),
        F.col("nombre_restaurante").alias("restaurante"),
        F.col("tipo_nombre").alias("tipo_pedido"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.round(F.sum("precio_total_pedido"), 2).alias("ingresos"),
    )
)

w_tipo = Window.partitionBy("restaurante", "tipo_pedido").orderBy("anio", "mes")
df_tipo = (
    df_tipo
    .withColumn("pedidos_ant", F.lag("total_pedidos", 1).over(w_tipo))
    .withColumn("crecimiento_pct",
        F.when(F.col("pedidos_ant") > 0,
            F.round((F.col("total_pedidos") - F.col("pedidos_ant"))
                    * 100.0 / F.col("pedidos_ant"), 2)
        ).otherwise(F.lit(None)))
    .drop("pedidos_ant")
    .orderBy("restaurante", "tipo_pedido", "anio", "mes")
)

df_tipo.show(20, truncate=False)

## 5. Guardar resultados

In [ ]:
import shutil, os

WAREHOUSE = "/opt/hive/data/warehouse/restaurant_dw.db"

def save_table(df, nombre):
    tabla = f"restaurant_dw.{nombre}"
    spark.sql(f"DROP TABLE IF EXISTS {tabla}")
    ruta = f"{WAREHOUSE}/{nombre}"
    if os.path.exists(ruta):
        shutil.rmtree(ruta, ignore_errors=True)
    df.write.mode("overwrite").saveAsTable(tabla)
    print(f"✅ {tabla}: {df.count()} filas guardadas.")

save_table(df_mom,    "resultado_crecimiento_mensual")
save_table(df_ytd,    "resultado_crecimiento_ytd")
save_table(df_cancel, "resultado_tasa_cancelacion_mensual")
save_table(df_tipo,   "resultado_crecimiento_tipo_pedido")

spark.stop()